# What This Walkthrough Does

This walkthrough introduces APIs through one concrete example: the Wikipedia /
MediaWiki API.

By the end, you should understand how to:

- send an API request from Python
- inspect the response before turning it into a table
- distinguish search results from page-level content
- follow pagination tokens
- collect more than one batch of results
- retrieve article text through the API
- collect external links and image metadata
- save raw data, processed data, and provenance information

The point is not just to "get data". The point is to understand how every API
request is also a research design choice.

::: {.callout-note}
This notebook mirrors the live coding file
`scripts/teaching_walkthroughs/01_api_collection_walkthrough.py`, but it is
written as a readable handout.
:::

# Imports and API Endpoint

We start by loading the Python packages used in this session.

## Packages Used in This Notebook

This walkthrough uses:

- `requests` to send HTTP requests to the MediaWiki API
- `pandas` to turn extracted rows into tables and save CSV files
- `Path` from `pathlib` to create file and folder paths safely
- `pprint` to print nested JSON in a readable way
- `time` to pause between repeated requests

The important distinction is that `requests` collects data, while `pandas` structures the data after collection.


In [1]:
# Path helps create file and folder paths in a way that works across operating systems.
from pathlib import Path
# pprint prints nested dictionaries/lists more readably than plain print().
from pprint import pprint
# time lets us pause between requests, which is important for respectful collection.
import time

# pandas is used after collection to make tables and save CSV files.
import pandas as pd
# requests sends HTTP requests to APIs and web pages.
import requests


`requests` sends HTTP requests.  
`pandas` turns lists of records into tables.  
`pprint` prints nested dictionaries and lists in a more readable way.

MediaWiki is useful for teaching because it is public, documented, and does not
require login credentials for basic queries.

## API Concepts Before the First Request

Before reading the code, keep these distinctions in mind:

| Concept | Meaning in this walkthrough |
|---|---|
| endpoint | the API address where the request is sent |
| parameters | the settings that define what we ask for |
| headers | metadata about the request, such as the User-Agent |
| response object | the HTTP-level object returned by `requests.get()` |
| payload | the parsed data inside the response, usually JSON |
| search result | a record saying that a page matched the query |
| page-level request | a second request asking for details about one known page |
| pagination | collecting results in batches rather than all at once |

These are methodological choices, not just technical details. The endpoint, parameters, page size, and stopping rule shape the dataset.

In [2]:
API_URL = "https://en.wikipedia.org/w/api.php"


The endpoint is the API address. By itself, it does not yet say what we want.
That is what parameters are for.

# Build One API Request

These parameters are sent after the `?` in the API URL. Together they define
what the API should do and what kind of data can come back.

## Where the MediaWiki API Parameters Are Documented

The parameters in this notebook are not invented by us. They come from the official MediaWiki API documentation.

Useful starting points:

- [MediaWiki API main page](https://www.mediawiki.org/wiki/API:Main_page): overview of the API and general concepts.
- [API:Query](https://www.mediawiki.org/wiki/API:Query): documentation for `action=query`, which is the main read-data action used here.
- [API:Search](https://www.mediawiki.org/wiki/API:Search): documentation for `list=search`, including parameters such as `srsearch`, `srlimit`, and `sroffset`.
- [API:Continue](https://www.mediawiki.org/wiki/API:Continue): explains continuation and pagination, including why APIs return tokens or offsets instead of all results at once.
- [Extension:TextExtracts](https://www.mediawiki.org/wiki/Extension:TextExtracts#API): documentation for `prop=extracts`, including `exintro`, `explaintext`, and `exchars`.
- [API help endpoint](https://en.wikipedia.org/w/api.php?action=help): machine-generated help from the live Wikipedia API.

When adapting this script, first look up the relevant module in the documentation, then decide which parameters fit the research question. Parameters are not just technical settings: they define what can enter the dataset.


In [23]:
params = {
    # action="query" tells MediaWiki that we want to read existing wiki data.
    "action": "query",

    # list="search" says that this query should return search results.
    "list": "search",

    # srsearch is the search string. This is a research design choice:
    # different search terms create different possible datasets.
    "srsearch": "digital services act",

    # srlimit is the number of search results requested per API response.
    # It is the batch size, not necessarily the total collection size.
    "srlimit": 5,

    # format="json" asks for machine-readable JSON.
    "format": "json",

    # formatversion=2 gives a simpler, more modern JSON structure.
    "formatversion": 2,
}


The request also includes a header. A `User-Agent` identifies the script. Many
APIs expect automated requests to identify themselves.

In [4]:
headers = {
    "User-Agent": "methodsNET-VLOP-course/1.0 teaching walkthrough",
}


Now we send the request.

In [24]:
# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(API_URL, params=params, headers=headers, timeout=30)


The object called `response` is not yet the dataset. It is the full HTTP
response object.

Useful things to inspect:

In [6]:
print(response.url)
print(response.status_code)
# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
print(response.headers.get("content-type"))


https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=digital+services+act&srlimit=50&format=json&formatversion=2
200
application/json; charset=utf-8


::: {.callout-tip}
The printed URL is useful because it shows exactly what Python sent to the API
after encoding the parameter dictionary.
:::

# Response Object vs Data Payload

The `response` object contains status information, headers, text, and other
metadata about the HTTP response.

To access the returned JSON data as Python dictionaries and lists, use:

In [25]:
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
payload = response.json()


Inspect the raw nested structure before flattening it into a table.

In [8]:
pprint(payload)


{'batchcomplete': True,
 'continue': {'continue': '-||', 'sroffset': 50},
 'query': {'search': [{'ns': 0,
                       'pageid': 65193550,
                       'size': 86121,
                       'snippet': 'The <span '
                                  'class="searchmatch">Digital</span> <span '
                                  'class="searchmatch">Services</span> <span '
                                  'class="searchmatch">Act</span> (DSA) is an '
                                  'EU regulation that entered into force in '
                                  '2022, establishing a comprehensive legal '
                                  'framework for <span '
                                  'class="searchmatch">digital</span> <span '
                                  'class="searchmatch">services</span> '
                                  'accountability',
                       'timestamp': '2026-06-20T03:14:50Z',
                       'title': 'Digital Services Act

Pause and identify:

- Which fields describe the API response?
- Which fields describe the search results?
- Where is the actual list of search-result records?

In this response, the search results are here:

In [26]:
# payload["query"]["search"] walks through nested JSON: payload -> query -> search-result list.
search_results = payload["query"]["search"]
print(type(search_results))
print(len(search_results))


<class 'list'>
5


Each item in this list is one search result.

In [ ]:
pprint(search_results[0]) # 0 because python starts counting from 0, not 1


{'ns': 0,
 'pageid': 65193550,
 'size': 86121,
 'snippet': 'The <span class="searchmatch">Digital</span> <span '
            'class="searchmatch">Services</span> <span '
            'class="searchmatch">Act</span> (DSA) is an EU regulation that '
            'entered into force in 2022, establishing a comprehensive legal '
            'framework for <span class="searchmatch">digital</span> <span '
            'class="searchmatch">services</span> accountability',
 'timestamp': '2026-06-20T03:14:50Z',
 'title': 'Digital Services Act',
 'wordcount': 7420}


Search results are not full Wikipedia articles. They are records about pages
that matched the search query.

# Inspect Available Fields

Before choosing columns to save in for a table/data frame, inspect the fields available in the search
result records.

In [10]:
for item in search_results[:3]:
    print(item.keys())


dict_keys(['ns', 'title', 'pageid', 'size', 'wordcount', 'snippet', 'timestamp'])
dict_keys(['ns', 'title', 'pageid', 'size', 'wordcount', 'snippet', 'timestamp'])
dict_keys(['ns', 'title', 'pageid', 'size', 'wordcount', 'snippet', 'timestamp'])


To see the set of all keys present across the records:

In [27]:
all_keys = set().union(*(item.keys() for item in search_results))
print(all_keys)


{'snippet', 'timestamp', 'wordcount', 'size', 'pageid', 'title', 'ns'}


# Flatten Search Results Into a Table

APIs often return nested data. For analysis, we often turn selected fields into
a rectangular table.

This is useful, but it is also lossy: the table contains only the fields we
decide to keep.

In [28]:
rows = []

for item in search_results:
    row = {
        # MediaWiki's stable numeric page identifier.
        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        "pageid": item.get("pageid"),

        # Human-readable page title.
        "title": item.get("title"),

        # Last-edit timestamp according to the search result record.
        "timestamp": item.get("timestamp"),

        # Rough size measure for the page.
        "wordcount": item.get("wordcount"),

        # Short HTML-marked search excerpt, not the full article text.
        "snippet": item.get("snippet"),
    }
    rows.append(row)


Now convert the list of dictionaries into a `pandas` DataFrame.

In [14]:
# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
df = pd.DataFrame(rows)
df.head()


,pageid,title,timestamp,wordcount,snippet
0,65193550,Digital Services Act,2026-06-20T03:14:50Z,7420,"The <span class=""searchmatch"">Digital</span> <..."
1,66230371,Digital Markets Act,2026-06-11T19:50:04Z,12016,"The <span class=""searchmatch"">Digital</span> M..."
2,295668,Content moderation,2026-06-23T23:47:33Z,3516,"<span class=""searchmatch"">Digital</span> <span..."
3,67646530,Online Safety Act 2023,2026-06-24T12:39:02Z,5945,"through &quot;<span class=""searchmatch"">servic..."
4,47652080,Digital Single Market,2026-06-22T19:07:06Z,6123,"Market For <span class=""searchmatch"">Digital</..."


Look at the snippets:

In [2]:
for snippet in df["snippet"].head():
    print(snippet)
    print()


NameError: name 'df' is not defined

::: {.callout-important}
The `snippet` field is not the article text. It is a search excerpt showing why
the page matched the query.
:::

# From Search Result to Wikipedia Page

A search result usually contains an identifier that lets us request more
information about the page.

For MediaWiki, the most useful identifier is `pageid`.

In [29]:
first_result = rows[0]

page_id = first_result["pageid"]
page_title = first_result["title"]

print("pageid:", page_id)
print("title:", page_title)


pageid: 65193550
title: Digital Services Act


The `pageid` can also be used to construct a normal browser URL:

In [30]:
page_browser_url = f"https://en.wikipedia.org/?curid={page_id}"
print(page_browser_url)


https://en.wikipedia.org/?curid=65193550


This URL is for human inspection in the browser. For reproducible data
collection, we usually prefer another API request.

# Retrieve Page-Level Content Through the API

Now we switch from searching for records to requesting properties of one known
page.

The important change is:

- `list="search"` returns search results matching a query
- `prop="info|extracts"` returns properties of a page we already identified

In [31]:
page_params = {
    "action": "query",

    # info gives page metadata; extracts gives text extracted from the article.
    "prop": "info|extracts",

    # Request one specific page by stable numeric page ID.
    "pageids": page_id,

    # Ask the info module to include the canonical page URL.
    "inprop": "url",

    # Ask for the lead section only. Remove this if you want a longer extract.
    "exintro": 1,

    # Ask for plain text instead of HTML.
    "explaintext": 1,

    # Limit the extract length for classroom inspection.
    "exchars": 1200,

    "format": "json",
    "formatversion": 2,
}


Send the page-level request:

In [ ]:
# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
page_response = requests.get(API_URL, params=page_params, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
page_response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
page_payload = page_response.json()


With `formatversion=2`, pages are returned as a list. Because we requested one
page, the first list element is the page record we need.

In [ ]:
page_record = page_payload["query"]["pages"][0]

print(page_response.url)
print(page_record.keys())


Inspect the page-level fields:

In [ ]:
# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
print("Title:", page_record.get("title"))
print("URL:", page_record.get("fullurl"))
print()
print(page_record.get("extract"))


::: {.callout-note}
This is still API collection, not web scraping. We are not parsing the
Wikipedia website HTML. We are asking the MediaWiki API for article content.
:::

# Pagination

Most APIs do not return all matching records at once. They return one batch at a
time.

In the MediaWiki search API:

- `srlimit` is the batch size
- `sroffset` is the starting position for the next batch
- `continue` tells the API how to continue

The first response includes a continuation object if there are more results.

In [17]:
# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
continuation = payload.get("continue", {})
print(continuation)


{'sroffset': 50, 'continue': '-||'}


To request the next batch, combine the original parameters with the continuation
parameters.

In [ ]:
# **continuation unpacks the API continuation fields into the request parameters for the next batch.
params_page_2 = {**params, **continuation}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response_2 = requests.get(API_URL, params=params_page_2, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response_2.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
payload_2 = response_2.json()


https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=digital+services+act&srlimit=50&format=json&formatversion=2&sroffset=50&continue=-%7C%7C


Check that the second batch contains different records.

In [22]:
# payload["query"]["search"] walks through nested JSON: payload -> query -> search-result list.
page_1_ids = [item["pageid"] for item in payload["query"]["search"]]
page_2_ids = [item["pageid"] for item in payload_2["query"]["search"]]

print("Page 1 IDs:", page_1_ids[:10])
print("Page 2 IDs:", page_2_ids[:10])
print("Overlap:", set(page_1_ids) & set(page_2_ids))

print(page_1_ids)


Page 1 IDs: [65193550, 66230371, 295668, 67646530, 47652080, 25286840, 20648089, 70137000, 18935388, 34092671]
Page 2 IDs: [57837019, 18581669, 67576350, 73917911, 331302, 41501305, 38104075, 64134773, 77813882, 196010]
Overlap: {331302}
[65193550, 66230371, 295668, 67646530, 47652080, 25286840, 20648089, 70137000, 18935388, 34092671, 68882246, 9933471, 39922603, 54174510, 1305348, 22757963, 708000, 76980362, 79397333, 77946617, 80542801, 68692470, 68294262, 67032801, 5897742, 47262459, 17728747, 82337203, 1649105, 61481948, 37082149, 51024759, 67241140, 63451675, 57174306, 4075736, 68774383, 56846326, 73295318, 59750806, 70232481, 58250, 74540504, 81103527, 83149340, 73236747, 43921488, 18934863, 59463, 331302]


::: {.callout-tip}
If `srlimit=50`, then `len(payload_2["query"]["search"])` will often be 50.
That only tells us the batch was full. The more informative check is whether
the IDs are different.
:::

# Collect Several Batches

The previous example collected one continuation step by hand. In real work, we
usually put this logic in a loop.

This version collects a fixed maximum number of batches. That makes the
stopping rule explicit.

In [ ]:
def collect_wikipedia_search(query, max_pages=3, page_size=50, pause_seconds=1):
    """Collect several batches of MediaWiki search results."""

    rows = []
    raw_pages = []
    continuation = {}

    for page_number in range(1, max_pages + 1):
        request_params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": page_size,
            "format": "json",
            "formatversion": 2,
            # **continuation unpacks the API continuation fields into the request parameters for the next batch.
            **continuation,
        }

        # requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
        response = requests.get(
            API_URL,
            params=request_params,
            headers=headers,
            timeout=30,
        )

        # raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
        response.raise_for_status()
        # .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
        page_payload = response.json()

        raw_pages.append(
            {
                "page_number": page_number,
                "request_url": response.url,
                "status_code": response.status_code,
                "payload": page_payload,
            }
        )

        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        for item in page_payload.get("query", {}).get("search", []):
            rows.append(
                {
                    "query": query,
                    "page_number": page_number,
                    "pageid": item.get("pageid"),
                    "title": item.get("title"),
                    "timestamp": item.get("timestamp"),
                    "wordcount": item.get("wordcount"),
                    "snippet": item.get("snippet"),
                }
            )

        print("Collected so far:", len(rows))

        continuation = page_payload.get("continue", {})
        if not continuation:
            break

        # time.sleep(...) pauses between requests so the script does not make rapid-fire calls to a server.
        time.sleep(pause_seconds)

    return rows, raw_pages


Run a small collection:

In [ ]:
search_rows, raw_pages = collect_wikipedia_search(
    query="digital services act",
    max_pages=2,
    page_size=25,
)

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
search_df = pd.DataFrame(search_rows)
search_df[["page_number", "pageid", "title", "wordcount"]].head()


::: {.callout-warning}
In live API work, requests can fail. Status `429` means "too many requests".
Good scripts should pause, retry carefully, and avoid collecting faster than the
API allows.
:::

# Collect Page Extracts for Search Results

Search results tell us which pages matched. To collect page-level article
content, we request each page by `pageid`.

Here is a small helper function for one page.

In [ ]:
def get_page_extract(pageid, extract_chars=1200):
    """Request page URL and plain-text extract for one Wikipedia page."""

    page_params = {
        "action": "query",
        "prop": "info|extracts",
        "pageids": pageid,
        "inprop": "url",
        "explaintext": 1,
        "exchars": extract_chars,
        "format": "json",
        "formatversion": 2,
    }

    # requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
    response = requests.get(API_URL, params=page_params, headers=headers, timeout=30)
    # raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
    response.raise_for_status()
    # .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
    page_payload = response.json()

    return page_payload["query"]["pages"][0]


Collect extracts for the first few search results:

In [ ]:
page_rows = []

for row in search_rows[:5]:
    page_record = get_page_extract(row["pageid"])

    page_rows.append(
        {
            # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
            "pageid": page_record.get("pageid"),
            "title": page_record.get("title"),
            "fullurl": page_record.get("fullurl"),
            "extract": page_record.get("extract"),
        }
    )

    # time.sleep(...) pauses between requests so the script does not make rapid-fire calls to a server.
    time.sleep(1)

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
pages_df = pd.DataFrame(page_rows)
pages_df.head()


This gives us a second table: one row per article page, with article-level
content.

# External Links

We can also use the API to ask which external links appear on a page.

In [ ]:
external_links_params = {
    "action": "query",
    "prop": "extlinks",
    "pageids": page_id,
    "ellimit": "max",
    "format": "json",
    "formatversion": 2,
}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
external_links_response = requests.get(
    API_URL,
    params=external_links_params,
    headers=headers,
    timeout=30,
)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
external_links_response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
external_links_payload = external_links_response.json()


Extract the external links:

In [ ]:
external_links_record = external_links_payload["query"]["pages"][0]
# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
external_links = external_links_record.get("extlinks", [])

external_urls = [
    link.get("url")
    for link in external_links
    if link.get("url")
]

external_links_row = {
    "pageid": external_links_record.get("pageid"),
    "page_title": external_links_record.get("title"),
    "external_links": external_urls,
    "external_link_count": len(external_urls),
}

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
external_links_df = pd.DataFrame([external_links_row])
external_links_df


This is a one-row-per-page format. The `external_links` column contains a list
of URLs.

# Image Metadata

Image collection also starts with the page ID.

First, ask which image/file titles are used on the page:

In [ ]:
image_params = {
    "action": "query",
    "prop": "images",
    "pageids": page_id,
    "imlimit": "max",
    "format": "json",
    "formatversion": 2,
}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
image_response = requests.get(API_URL, params=image_params, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
image_response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
image_payload = image_response.json()

image_record = image_payload["query"]["pages"][0]
# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
images = image_record.get("images", [])
image_titles = [image["title"] for image in images]

image_titles[:10]


Then ask for metadata about those files:

In [ ]:
if image_titles:
    image_info_params = {
        "action": "query",
        "prop": "imageinfo",
        "titles": "|".join(image_titles),
        "iiprop": "url|size|mime",
        "format": "json",
        "formatversion": 2,
    }

    # requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
    image_info_response = requests.get(
        API_URL,
        params=image_info_params,
        headers=headers,
        timeout=30,
    )
    # raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
    image_info_response.raise_for_status()
    # .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
    image_info_payload = image_info_response.json()

    image_rows = []

    for file_page in image_info_payload["query"]["pages"]:
        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        info = file_page.get("imageinfo", [{}])[0]
        image_rows.append(
            {
                "file_title": file_page.get("title"),
                "image_url": info.get("url"),
                "mime": info.get("mime"),
                "width": info.get("width"),
                "height": info.get("height"),
                "size": info.get("size"),
            }
        )

    # pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
    image_df = pd.DataFrame(image_rows)
else:
    image_df = pd.DataFrame(
        columns=["file_title", "image_url", "mime", "width", "height", "size"]
    )

image_df.head()


::: {.callout-note}
Downloading image files is a separate methodological and practical decision. It
creates larger files and can raise licensing, storage, and ethics questions.
For Day 1, metadata and URLs are enough unless there is extra time.
:::

# Save Raw, Processed, and Provenance Files

Good data collection workflows distinguish between:

- raw data: what the API returned
- processed data: the table we created
- provenance: documentation of where the data came from and how it was created

Create output folders:

In [ ]:
outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

# mkdir(..., parents=True, exist_ok=True) creates the folder and any missing parent folders, without failing if it already exists.
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)


Define output paths:

In [ ]:
raw_path = raw_dir / "notebook_wikipedia_raw.json"
search_processed_path = processed_dir / "notebook_wikipedia_search_results.csv"
pages_processed_path = processed_dir / "notebook_wikipedia_page_extracts.csv"
provenance_path = reports_dir / "notebook_wikipedia_provenance.json"


Save the outputs:

In [ ]:
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(raw_pages).to_json(raw_path, orient="values", indent=2)
# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
search_df.to_csv(search_processed_path, index=False)
pages_df.to_csv(pages_processed_path, index=False)


Write a small provenance record:

In [ ]:
provenance = {
    "source": "MediaWiki API",
    "endpoint": API_URL,
    "query": "digital services act",
    "page_size": 25,
    "max_pages": 2,
    "raw_output": str(raw_path),
    "search_results_output": str(search_processed_path),
    "page_extracts_output": str(pages_processed_path),
    "method_note": (
        "Search API results are not a complete measure of platform attention, "
        "public importance, or article relevance."
    ),
}

# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(provenance).to_json(provenance_path, indent=2)


Check what was saved:

In [ ]:
print(raw_path)
print(search_processed_path)
print(pages_processed_path)
print(provenance_path)


# Exercise

Choose one of the following adaptations:

1. Change the search query.
2. Change `page_size` and `max_pages`.
3. Collect page extracts for more search results.
4. Save external links for several pages rather than only one.
5. Add one new column to the processed search-results table.

For your adaptation, answer:

- What did you change?
- Why is this change substantively meaningful?
- What new records or fields can now enter the dataset?
- What still remains invisible?

# Key Takeaways

APIs are structured access routes. They make some data easy to collect and other
data hard or impossible to access.

For every API dataset, ask:

- What did the endpoint allow me to request?
- What did my parameters select?
- What identifiers connect search results to detailed records?
- How did pagination shape the final collection?
- What did I save as raw evidence?
- What provenance information would someone need to understand or reproduce the
  dataset?